In [1]:
!pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 833.7 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.6/201.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 8.0 MB/s eta

In [2]:
import re

# Read and parse the emotions.txt file
def parse_emotions_file(file_path):
    emotion_data = {}
    current_emotion = None
    current_description = []

    with open(file_path, 'r') as file:
        lines = file.readlines()
        for line in lines:
            line = line.strip()
            # Skip empty lines
            if not line:
                continue
            # Check for octant headers (e.g., "Octant 1: High Valence, High Arousal, High Dominance")
            if line.startswith("Octant"):
                continue
            # Check for emotion name (single word or short phrase, not starting with "Enhanced")
            if line and not line.startswith("Enhanced") and not re.match(r"^[A-Za-z\s]+$", line):
                continue
            if line and not line.startswith("Enhanced"):
                # Save previous emotion's description if exists
                if current_emotion and current_description:
                    emotion_data[current_emotion] = " ".join(current_description)
                    current_description = []
                current_emotion = line
            elif line and current_emotion:
                # Accumulate description lines
                current_description.append(line)

        # Save the last emotion's description
        if current_emotion and current_description:
            emotion_data[current_emotion] = " ".join(current_description)

    return emotion_data

# Parse the uploaded file
emotion_data = parse_emotions_file('/content/rag.txt')
print("Parsed Emotions:", list(emotion_data.keys()))

Parsed Emotions: ['Happy', 'Excited', 'Confident', 'Enthusiastic', 'Proud', 'Thrilled', 'Delighted', 'Overjoyed', 'Giddy', 'Exhilarated', 'Calm', 'Content', 'Satisfied', 'Secure', 'At Ease', 'Affectionate', 'Fond', 'Warm', 'Tender', 'Sentimental', 'Angry', 'Frustrated', 'Annoyed', 'Disgusted', 'Contemptuous', 'Scared', 'Nervous', 'Worried', 'Frightened', 'Alarmed', 'Disdainful', 'Scornful', 'Condescending', 'Dismissive', 'Supercilious', 'Sad', 'Depressed', 'Melancholy', 'Despondent', 'Hopeless']


In [4]:
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize the Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize ChromaDB client with persistent storage
client = chromadb.PersistentClient(path="./emotion_db")

# Create or get a collection
collection = client.get_or_create_collection(name="emotions")

In [5]:
# Prepare data for ChromaDB
documents = []
metadatas = []
ids = []
for idx, (emotion, description) in enumerate(emotion_data.items()):
    documents.append(description)
    metadatas.append({"emotion": emotion})
    ids.append(f"emotion_{idx}")

# Generate embeddings for the descriptions
embeddings = model.encode(documents, convert_to_tensor=False).tolist()

# Add data to the ChromaDB collection
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

# Persist the database (handled automatically by PersistentClient)

In [9]:
import os
from google.colab import files
# Zip the database directory
!zip -r emotion_db.zip ./emotion_db

# Download the zipped database
files.download('emotion_db.zip')

updating: emotion_db/ (stored 0%)
updating: emotion_db/chroma.sqlite3 (deflated 65%)
updating: emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/ (stored 0%)
updating: emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/link_lists.bin (stored 0%)
updating: emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/header.bin (deflated 61%)
updating: emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/length.bin (deflated 100%)
updating: emotion_db/ca9dd079-d065-4a8e-b8a5-300bd196429f/data_level0.bin (deflated 100%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
from google.colab import output

# Function to query ChromaDB and display results
def query_emotion_knowledge_base(emotion):
    # Encode the input emotion as a query
    query_embedding = model.encode([emotion], convert_to_tensor=False).tolist()

    # Query the collection
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=1  # Retrieve the top matching result
    )

    # Extract and display results
    if results['metadatas'][0]:
        emotion_name = results['metadatas'][0][0]['emotion']
        description = results['documents'][0][0]
        distance = results['distances'][0][0]

        print(f"Queried Emotion: {emotion}")
        print(f"Matched Emotion: {emotion_name}")
        print(f"Similarity Score (Distance): {distance:.4f}")
        print("\nDescription:")
        print(description)
    else:
        print(f"No results found for emotion: {emotion}")

# Input prompt for emotion
emotion_input = input("Enter an emotion to query (e.g., Happy, Excited): ")

# Clear previous output for cleaner display
output.clear()

# Query and display
query_emotion_knowledge_base(emotion_input)

Queried Emotion: giddy
Matched Emotion: Giddy
Similarity Score (Distance): 1.6712

Description:
Enhanced Therapeutic Approach: Channel playful dizziness through whimsical music that maintains emotional safety while allowing carefree expression. Use major keys with chromatic passing tones and unexpected harmonic turns, tempos between 110-140 BPM with occasional rubato and playful accelerandos. Choose instruments with unique timbral characteristics: accordion, banjo, piccolo, and muted brass with plunger effects. Dynamic levels should vary frequently between piano and forte with sudden changes and echo effects for playful surprise. Harmonic progressions should include deceptive cadences (V-vi instead of V-I) and circle progressions that create harmonic adventure while maintaining tonal stability. Use irregular phrase lengths (5 or 7 measures instead of 4 or 8) and mixed meters (alternating 4/4 and 3/4) to create rhythmic playfulness. Melodic lines should feature wide interval leaps, glis

In [15]:
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize ChromaDB client with persistent storage
client = chromadb.PersistentClient(path="./emotion_db")

# Get the emotions collection
collection = client.get_or_create_collection(name="emotions")

# Get the total number of entries in the collection
count = collection.count()

# Get emotion names from metadata to verify content
metadata = collection.get(include=["metadatas"])["metadatas"]
emotions = [item["emotion"] for item in metadata] if metadata else []

# Infer embedding dimensionality from the Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = model.get_sentence_embedding_dimension()

# Display results
print(f"Knowledge Base Dimensions:")
print(f"Number of Entries: {count}")
print(f"Embedding Dimensionality: {embedding_dim}")
print(f"Stored Emotions: {emotions if emotions else 'None'}")

Knowledge Base Dimensions:
Number of Entries: 40
Embedding Dimensionality: 384
Stored Emotions: ['Happy', 'Excited', 'Confident', 'Enthusiastic', 'Proud', 'Thrilled', 'Delighted', 'Overjoyed', 'Giddy', 'Exhilarated', 'Calm', 'Content', 'Satisfied', 'Secure', 'At Ease', 'Affectionate', 'Fond', 'Warm', 'Tender', 'Sentimental', 'Angry', 'Frustrated', 'Annoyed', 'Disgusted', 'Contemptuous', 'Scared', 'Nervous', 'Worried', 'Frightened', 'Alarmed', 'Disdainful', 'Scornful', 'Condescending', 'Dismissive', 'Supercilious', 'Sad', 'Depressed', 'Melancholy', 'Despondent', 'Hopeless']


In [38]:
!pip install chromadb sentence-transformers transformers torch

In [39]:
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import torch

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="./emotion_db")
collection = chroma_client.get_or_create_collection(name="emotions")

# Initialize Sentence Transformer for querying
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize Flan-T5 for emotion detection (local, no API)
device = 0 if torch.cuda.is_available() else -1  # Use GPU if available
llm = pipeline("text2text-generation", model="google/flan-t5-small", device=device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [40]:
def detect_emotion_and_retrieve(vad_scores):
    valence, arousal, dominance = vad_scores

    # Concise prompt with VAD definitions (1-5 scale)
    vad_prompt = f"""
    VAD scores (1-5):
    - Valence: {valence} (1=very unpleasant, 5=very pleasant)
    - Arousal: {arousal} (1=very calm, 5=very energetic)
    - Dominance: {dominance} (1=very submissive, 5=very dominant)
    Predict the emotion (e.g., Happy, Sad). Respond with only the emotion name.
    Example: Valence=5, Arousal=4, Dominance=4 → Happy
    Example: Valence=2, Arousal=2, Dominance=2 → Sad
    """

    # Detect emotion using Flan-T5
    response = llm(vad_prompt, max_length=10, num_return_sequences=1)[0]["generated_text"]
    detected_emotion = response.strip()

    # Query ChromaDB for the closest emotion description
    query_embedding = embedder.encode([detected_emotion], convert_to_tensor=False).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=1)

    # Extract results
    if results['metadatas'][0]:
        matched_emotion = results['metadatas'][0][0]['emotion']
        description = results['documents'][0][0]
        distance = results['distances'][0][0]
    else:
        matched_emotion, description, distance = "None", "No match found", float('inf')

    return detected_emotion, matched_emotion, description, distance

def generate_musicgen_prompt(detected_emotion, matched_emotion, description, vad_scores):
    valence, arousal, dominance = vad_scores
    prompt = f"""
    Compose therapeutic music for '{detected_emotion}' (closest: '{matched_emotion}').
    VAD: Valence={valence}, Arousal={arousal}, Dominance={dominance}.
    Description: {description}
    Use suitable tempo, key, and instruments.
    """
    return prompt

In [43]:
from google.colab import output

# Input VAD scores (1 to 5 scale)
valence = float(input("Enter Valence score (1 to 5): "))
arousal = float(input("Enter Arousal score (1 to 5): "))
dominance = float(input("Enter Dominance score (1 to 5): "))
vad_scores = [valence, arousal, dominance]

# Validate input
if not all(1 <= x <= 5 for x in vad_scores):
    print("Error: VAD scores must be between 1 and 5.")
else:
    # Clear previous output
    output.clear()

    # Detect emotion, retrieve description, and generate prompt
    detected_emotion, matched_emotion, description, distance = detect_emotion_and_retrieve(vad_scores)
    musicgen_prompt = generate_musicgen_prompt(detected_emotion, matched_emotion, description, vad_scores)

    # Display results
    print(f"Detected Emotion: {detected_emotion}")
    print(f"Closest Emotion in Knowledge Base: {matched_emotion}")
    print(f"Similarity Score (Distance): {distance:.4f}")
    print(f"\nKnowledge Base Description:\n{description}")
    print(f"\nMusicGen Prompt:\n{musicgen_prompt}")

Both `max_new_tokens` (=256) and `max_length`(=10) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Detected Emotion: Arousal
Closest Emotion in Knowledge Base: Scared
Similarity Score (Distance): 1.2290

Knowledge Base Description:
Enhanced Therapeutic Approach: Address fear through music that gradually builds safety while acknowledging emotional reality. Begin with music that validates fear using minor keys with diminished chords and unsettled rhythmic patterns, then gradually transition to major keys with stable harmonic progressions. Start with tempos around 100-120 BPM that match elevated heart rate, gradually slowing to 70-80 BPM as safety develops. Choose instruments that move from edgy to warm: begin with muted strings and woodwinds in low registers, gradually adding warmer timbres like cello and acoustic guitar. Dynamic levels should start in the mezzo-forte range with some sudden changes, gradually settling into consistent piano to mezzo-piano levels that promote calm. Harmonic progressions should begin with unresolved tensions and ambiguous tonality, gradually moving towar

In [ ]:
from transformers import pipeline
import scipy

synthesiser = pipeline("text-to-audio", "facebook/musicgen-small")

music = synthesiser("lo-fi music with a soothing melody", forward_params={"do_sample": True})

scipy.io.wavfile.write("musicgen_out.wav", rate=music["sampling_rate"], data=music["audio"])


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]